# Solar filament segmentation: the improved pipeline

The code is in the package `improved/` (data, model, scoring, training, evaluation).
This notebook only runs it. Only the Kaggle data in `data/` is used.

Use the kernel **COS711 (venv, AMD GPU)**. Switch a flag to `True` to run that step;
results are saved under `outputs/`.

In [ ]:
import torch

from improved.config import EVAL, MODELS
from improved.data import analyse_data, prepare_cache
from improved.evaluation import evaluate, predict_test, size_report
from improved.model import pick_device
from improved.training import make_config, train

print("device:", pick_device(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")

RUN_PREPARE_CACHE = False  # about 2 minutes, once
RUN_ANALYSE_DATA = False   # facts about the data
RUN_TRAIN = False          # about 33 minutes
RUN_EVALUATE = False       # score on the validation fold and tune the clean-up
RUN_PREDICT_TEST = False   # predict the 180 test images
RUN_SIZE_REPORT = False    # size of the saved models

## 1. Data

Convert the data to fast files once, then look at what the data tells us.

In [ ]:
if RUN_PREPARE_CACHE:
    prepare_cache()

if RUN_ANALYSE_DATA:
    analyse_data()

## 2. Train

Change any setting by name, e.g. `make_config("improved", act="relu")` switches one idea off.
`make_config("baseline")` reproduces the baseline.

In [ ]:
if RUN_TRAIN:
    train(make_config("improved", fold=0), name="improved_f0")

## 3. Score on the validation fold

In [ ]:
if RUN_EVALUATE:
    evaluate(MODELS / "improved_f0.pt", tune=True, tta=True, tag="improved_f0_tta")

## 4. Predict the test images and report model sizes

In [ ]:
if RUN_PREDICT_TEST:
    predict_test(MODELS / "improved_f0.pt", EVAL / "improved_f0_tta.json", tta=True)

if RUN_SIZE_REPORT:
    size_report()